# 로컬 LLM 서버 연결 (Local LLM Connectivity)
## 로컬 LLM 서버 · OpenAI 호환 프로토콜 — Ollama(GPU) · llama.cpp(CPU) 두 경로

> 📦 **환경 설치·실행 명령**은 [`env_guides/M02_1_local_llm.md`](env_guides/M02_1_local_llm.md) 에 정리되어 있습니다.
> 반복·공통 코드는 [`agentic_lib/`](agentic_lib) 라이브러리(bootstrap·tools)로 분리해 두었습니다.

---

### 전제조건
- Windows 11 + **CMD(`cmd.exe`)** + Python **3.11**(`uv` 관리), 커널 = **`Agentic AI (uv)`**
- 기본 LLM = **로컬 Ollama + `qwen3:8b`** (클라우드 `google`(Gemini)은 비교/폴백용)
- 공통 1회 준비(uv 설치 → `uv venv` → `uv sync` → 커널 등록)는 [`env_guides/README.md`](env_guides/README.md) 참고

### 학습 목표
이 노트북은 `.env` 의 `LLM_PROVIDER` 설정과 **무관하게**, 두 로컬 경로를 **각각 명시적으로** 실행합니다.
1. **경로 1 — Ollama (GPU)**: `qwen3:8b` 서버 연결 후 (1) 연결 테스트 · (2) OpenAI 호환 응답 테스트
2. **경로 2 — llama.cpp (CPU)**: 소형 GGUF 를 CPU 로 서빙 후 (1) 연결 테스트 · (2) OpenAI 호환 응답 테스트

두 서버 모두 **OpenAI 호환 `/v1`** 이라, provider 이름만 바꿔 **동일한 `ChatOpenAI` 코드**로 테스트합니다.

### 아키텍처 개요
```
                        ┌─→ [경로 1] Ollama 서버 (GPU)     · :11434/v1  · qwen3:8b
[LangChain ChatOpenAI] ─┤
                        └─→ [경로 2] llama.cpp 서버 (CPU)  · :8000/v1   · 소형 GGUF
        (둘 다 OpenAI 호환 /v1 — provider 만 바꾸면 코드는 그대로)
```


In [1]:
# [setup] 자기완결적 셋업 — 이 노트북만 단독으로 실행 가능하게 한다
import sys, os
sys.path.insert(0, os.path.abspath(''))   # notebooks/ 를 import 경로에 추가

import utils
utils.reload_env()   # .env 재로드 (LLM_PROVIDER 등 갱신) + 현재 공급자 상태 출력

from utils import get_llm, print_provider_status

# 반복/공통 코드는 agentic_lib 라이브러리로 분리 (Part A 는 bootstrap·tools 만 사용)
from agentic_lib import bootstrap, tools
from agentic_lib.bootstrap import to_text   # 공급자 무관 응답 정규화(<think> 제거 포함)

# 로컬(ollama/llama.cpp)·클라우드 모두 OpenAI 호환 → 동일 인터페이스
utils.uv_install(['langchain', 'langchain-openai', 'langchain-community',
                  'langchain-google-genai', 'langchain-anthropic',
                  'openai', 'requests'])

print_provider_status()

LLM 공급자: nvidia
  NVIDIA build Key: 설정됨  /  Model: meta/llama-3.1-8b-instruct


[uv] 설치 완료: ['langchain', 'langchain-openai', 'langchain-community', 'langchain-google-genai', 'langchain-anthropic', 'openai', 'requests']
LLM 공급자: nvidia
  NVIDIA build Key: 설정됨  /  Model: meta/llama-3.1-8b-instruct


## 1. 환경 설정 — 로컬 LLM 공급자 (OpenAI 호환 프로토콜)

이 모듈은 **로컬 LLM 서버 연결**을 실습합니다. 로컬 서버도 클라우드 LLM 과 동일하게
**OpenAI 호환 API**(`/v1/chat/completions`)를 제공하므로, LangChain 의 `ChatOpenAI` 로 똑같이 연결됩니다.
즉 **공급자만 바꾸면 코드 변경 없이** 클라우드 ↔ 로컬, GPU ↔ CPU 를 오갈 수 있습니다.

### 이 노트북의 진행 방식 (`.env` 와 무관)
아래 §2·§3 에서 **두 로컬 경로를 각각 명시적으로** 실행합니다. `.env` 의 `LLM_PROVIDER` 값이
무엇이든, `get_llm("ollama")` / `get_llm("llamacpp")` 처럼 **provider 를 코드에서 직접 지정**하므로
설정에 좌우되지 않습니다.

| 경로 | 공급자 | 실행 | 모델 | 엔드포인트 |
|---|---|---|---|---|
| **경로 1** | `ollama` | **GPU**(자동 가속) | `qwen3:8b` | `:11434/v1` |
| **경로 2** | `llamacpp` | **CPU**(고정) | 소형 GGUF | `:8000/v1` |
| (참고) vLLM | `vllm` | Linux/GPU | — | Windows 비권장 |

각 경로마다 **(1) 연결 테스트**(OpenAI 호환 `/v1/models`)와 **(2) 응답 테스트**(`ChatOpenAI.invoke`)를 수행합니다.

> 설치·문제해결은 [`env_guides/M02_1_local_llm.md`](env_guides/M02_1_local_llm.md) 참고.

In [2]:
# [공통 헬퍼] OpenAI 호환 로컬 서버 테스트 — (1) 연결 테스트 · (2) 프로토콜 응답 테스트
# 이 노트북은 .env 의 LLM_PROVIDER 와 무관하게, 아래 두 경로를 '각각 명시적으로' 실행한다:
#   · 경로 1: Ollama (GPU)    — qwen3:8b        · :11434/v1
#   · 경로 2: llama.cpp (CPU) — 소형 GGUF        · :8000/v1
# 두 서버 모두 OpenAI 호환 /v1 이라, provider 이름만 바꿔 같은 코드로 테스트한다.
import requests
from langchain_core.messages import HumanMessage, SystemMessage

def check_openai_server(base_url, timeout=3.0):
    """OpenAI 호환 /v1/models 로 서버 헬스체크 + 서빙 모델 목록을 반환한다."""
    try:
        resp = requests.get(f"{base_url}/models", timeout=timeout)
        resp.raise_for_status()
        return True, [m["id"] for m in resp.json().get("data", [])]
    except Exception as e:
        return False, str(e)

def connection_test(provider, base_url):
    """(1) 로컬 LLM '연결 테스트' — OpenAI 호환 /v1/models 엔드포인트로 서버 응답을 확인한다."""
    print(f"[{provider}] 연결 테스트 → {base_url}/models")
    ok, info = check_openai_server(base_url)
    print("  " + (f"✅ 서버 응답 OK — 서빙 모델: {info}" if ok else f"❌ 연결 실패: {info}"))
    return ok

def response_test(provider):
    """(2) 'OpenAI 호환 프로토콜 응답 테스트' — get_llm(provider) 의 ChatOpenAI 로 실제 추론한다.

    provider 를 인자로 직접 지정하므로 .env 의 LLM_PROVIDER 와 무관하게 원하는 로컬 서버로 접속한다.
    로컬(ollama/llamacpp)도 클라우드와 동일한 ChatOpenAI 인터페이스라 코드가 같다.
    """
    llm = get_llm(provider)                       # setup 셀에서 import 한 utils.get_llm
    print(f"[{provider}] 응답 테스트 · LLM 타입={type(llm).__name__}")
    demo = llm.invoke([
        SystemMessage(content="당신은 간결하게 답하는 한국어 어시스턴트입니다."),
        HumanMessage(content="로컬 LLM 을 OpenAI 호환 API 로 쓰는 장점을 한 문장으로 설명해줘."),
    ])
    print(f"  응답: {to_text(demo.content)}")      # to_text: 공급자 무관 정규화(<think> 제거 등)
    return llm

print("헬퍼 준비 완료 — connection_test(provider, base_url) · response_test(provider)")

헬퍼 준비 완료 — connection_test(provider, base_url) · response_test(provider)


---
## 2. 경로 1 — Ollama (GPU)

첫 번째 경로는 **Ollama + `qwen3:8b`** 입니다. Ollama 는 백그라운드 서비스로 상주하며
**OpenAI 호환 엔드포인트**(`:11434/v1`)를 제공하고, **NVIDIA GPU 가 있으면 자동으로 GPU 가속**을 사용합니다.
`qwen3:8b` 는 네이티브 `tool_calls`(도구 호출)도 안정적으로 생성합니다.

```bat
winget install --id Ollama.Ollama -e   REM 설치(1회)
ollama serve                           REM 서버 실행(자동 실행 안 되면 수동)
ollama pull qwen3:8b                    REM 모델 다운로드(약 5GB, 1회)
curl.exe http://localhost:11434/api/tags   REM 동작 확인
```

진행: **(준비 확인)** → **(1) 연결 테스트** → **(2) OpenAI 호환 응답 테스트**

In [3]:
# [경로 1: Ollama · GPU] GPU 가속 안내
# Ollama 는 NVIDIA GPU 가 있으면 자동으로 GPU 가속을 사용한다(별도 설정 불필요).
import shutil
print("GPU(NVIDIA):", "감지됨 — Ollama 가 자동으로 GPU 가속" if shutil.which("nvidia-smi") else "없음 → CPU 로 실행")
print("서버/모델 준비 상태는 아래 연결 테스트(connection_test) 셀에서 확인합니다.")

GPU(NVIDIA): 감지됨 — Ollama 가 자동으로 GPU 가속
서버/모델 준비 상태는 아래 연결 테스트(connection_test) 셀에서 확인합니다.


In [4]:
# [경로 1] (1) Ollama 연결 테스트 — OpenAI 호환 /v1/models
connection_test("ollama", utils.OLLAMA_BASE_URL)

[ollama] 연결 테스트 → http://localhost:11434/v1/models


  ✅ 서버 응답 OK — 서빙 모델: ['qwen2.5:0.5b', 'nomic-embed-text:latest', 'qwen3:8b']


True

In [5]:
# [경로 1] (2) Ollama OpenAI 호환 프로토콜 응답 테스트 (GPU · qwen3:8b)
ollama_llm = response_test("ollama")   # get_llm("ollama") → ChatOpenAI(:11434/v1)

[ollama] 응답 테스트 · LLM 타입=ChatOpenAI


  응답: 로컬 LLM을 OpenAI 호환 API로 사용하면 데이터 프라이버시 보호와 비용 효율성을 동시에 달성할 수 있습니다.


---
## 3. 경로 2 — llama.cpp (CPU)

두 번째 경로는 **llama.cpp(`llama-cpp-python`)를 CPU 로** 직접 서빙하는 방법입니다.
GPU 가속은 경로 1(Ollama)이 담당하므로, 대비를 위해 여기서는 **의도적으로 CPU** 로 띄웁니다.
llama.cpp 서버도 **OpenAI 호환 `/v1`** 을 제공하므로, 경로 1 과 **완전히 같은 방식**으로
(1) 연결 · (2) 응답 테스트를 합니다. 컴파일러 없이 **사전 빌드 휠**로 설치합니다.

### 절차
**(0)** 백엔드=CPU·모델 선택 → **(1)** 휠 설치 → **(2)** GGUF 다운로드 → **(3)** 서버 기동(백그라운드)
→ **(1) 연결 테스트** → **(2) OpenAI 호환 응답 테스트**

```bat
:: (1) 설치 — CPU 사전 빌드 휠
uv pip install "llama-cpp-python[server]" --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu
:: (3) 서버 기동 — CPU 이므로 --n_gpu_layers 0
python -m llama_cpp.server --model <GGUF경로> --model_alias llamacpp --host 0.0.0.0 --port 8000 --n_gpu_layers 0 --n_ctx 4096
:: (4) 엔드포인트 확인
curl.exe http://localhost:8000/v1/models
```

> **vLLM 은?** Linux/GPU 권장. Windows 에서는 Docker(WSL2) 필요 + V2 Model Runner UVA 문제로 본 실습에서는 사용하지 않습니다.

### 3-(0) 백엔드=CPU 고정 · 모델 선택

이 경로는 **CPU 전용**입니다. 아래 셀은 참고용으로 GPU 유무와 GGUF 모델 카탈로그를 보여주되,
백엔드는 **`cpu` 로 고정**하고 CPU 에서 빠른 **소형 모델**(`qwen2.5-0.5b`)을 기본 선택합니다.
(더 큰 CPU 모델을 원하면 `MODEL_KEY` 만 바꾸세요 — 속도는 느려집니다.)

In [6]:
# [백엔드/모델 선택] 경로 2 는 'llama.cpp = CPU' 로 고정 (GPU 는 경로 1 Ollama 가 담당)
import shutil

# 1) 로드 가능한 모델 카탈로그 (모두 단일 파일 GGUF · 게이트 없음 · Q4_K_M 양자화)
MODEL_CATALOG = {
    "qwen2.5-0.5b": dict(repo="Qwen/Qwen2.5-0.5B-Instruct-GGUF",
                         file="qwen2.5-0.5b-instruct-q4_k_m.gguf", size="~0.4GB",
                         tier="cpu", note="초소형 · CPU 매우 빠름"),
    "qwen2.5-3b":   dict(repo="Qwen/Qwen2.5-3B-Instruct-GGUF",
                         file="qwen2.5-3b-instruct-q4_k_m.gguf", size="~2.0GB",
                         tier="cpu", note="CPU 가능 · 균형"),
    "llama3.1-8b":  dict(repo="bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
                         file="Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf", size="~4.9GB",
                         tier="gpu", note="GPU 권장 (CPU 는 느림)"),
}

# 2) 참고용 GPU 감지 — 이 경로는 CPU 로 고정하므로 '선택'에는 영향이 없다(정보 표시용).
gpu_available = shutil.which("nvidia-smi") is not None
print(f"GPU(NVIDIA) 참고: {'있음 → 경로1 Ollama 가 GPU 사용' if gpu_available else '없음'}")

# 3) 카탈로그 출력
print("\n로드 가능한 모델 (key | 크기 | 저장소):")
for k, m in MODEL_CATALOG.items():
    print(f"  - {k:13s} {m['size']:>7s}  {m['repo']}  · {m['note']}")

# 4) CPU 경로 고정값 — CPU 에서 빠른 소형 모델
BACKEND   = "cpu"            # 사전 빌드 CPU 휠 (컴파일러 불필요). 이 경로는 항상 CPU.
MODEL_KEY = "qwen2.5-0.5b"   # CPU 에서 매우 빠른 초소형 모델
print(f"\n선택됨(CPU 고정) → BACKEND={BACKEND} · MODEL_KEY={MODEL_KEY} ({MODEL_CATALOG[MODEL_KEY]['file']})")
print("더 큰 CPU 모델을 원하면 MODEL_KEY 를 'qwen2.5-3b' 로 바꾸세요(속도 느려짐).")

GPU(NVIDIA) 참고: 있음 → 경로1 Ollama 가 GPU 사용

로드 가능한 모델 (key | 크기 | 저장소):
  - qwen2.5-0.5b   ~0.4GB  Qwen/Qwen2.5-0.5B-Instruct-GGUF  · 초소형 · CPU 매우 빠름
  - qwen2.5-3b     ~2.0GB  Qwen/Qwen2.5-3B-Instruct-GGUF  · CPU 가능 · 균형
  - llama3.1-8b    ~4.9GB  bartowski/Meta-Llama-3.1-8B-Instruct-GGUF  · GPU 권장 (CPU 는 느림)

선택됨(CPU 고정) → BACKEND=cpu · MODEL_KEY=qwen2.5-0.5b (qwen2.5-0.5b-instruct-q4_k_m.gguf)
더 큰 CPU 모델을 원하면 MODEL_KEY 를 'qwen2.5-3b' 로 바꾸세요(속도 느려짐).


In [7]:
# [설치] llama.cpp 서버 — BACKEND 에 맞는 사전 빌드 휠 (컴파일러 불필요)
# BACKEND 는 위 '모델/백엔드 확인' 셀에서 설정됨 (cpu | cu121 | cu124)
WHEEL_INDEX = {
    "cpu":   "https://abetlen.github.io/llama-cpp-python/whl/cpu",
    "cu121": "https://abetlen.github.io/llama-cpp-python/whl/cu121",   # CUDA 12.1
    "cu124": "https://abetlen.github.io/llama-cpp-python/whl/cu124",   # CUDA 12.4 (최신 드라이버 하위호환)
}

idx = WHEEL_INDEX.get(BACKEND)
if idx:
    # --reinstall-package: 백엔드 전환(cpu↔cu124) 시 같은 버전이라도 휠을 다시 받도록 강제
    utils.run_cmd(f'uv pip install --reinstall-package llama-cpp-python "llama-cpp-python[server]" --extra-index-url {idx}')
elif BACKEND == "cu130":
    # cu130 사전 빌드 휠은 (Windows) 제공되지 않음 → 두 가지 선택지 안내
    print("[안내] cu130 사전 빌드 휠은 Windows 에 없습니다.")
    print("  방법 A) 권장: BACKEND='cu124' 로 변경 — 최신 NVIDIA(CUDA 13) 드라이버는 cu124 휠과 하위호환됩니다.")
    print("  방법 B) 소스 빌드 (CUDA Toolkit 13.0 + Visual Studio C++ 빌드도구 필요):")
    print("         set CMAKE_ARGS=-DGGML_CUDA=on")
    print('         uv pip install "llama-cpp-python[server]" --no-binary llama-cpp-python')
else:
    print(f"[오류] 알 수 없는 BACKEND={BACKEND!r}. cpu | cu121 | cu124 중에서 선택하세요.")


$ uv pip install --reinstall-package llama-cpp-python "llama-cpp-python[server]" --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu


Using Python 3.11.15 environment at: C:\Users\stshin\Documents\GitHub\Agentic AI Tutorial\.venv


Resolved 24 packages in 1.43s
Prepared 1 package in 40ms
Uninstalled 1 package in 63ms


Installed 1 package in 214ms
 ~ llama-cpp-python==0.3.34


In [8]:
# [모델] 선택된 GGUF 모델 다운로드 (위 카탈로그의 MODEL_KEY 기준)
import os
utils.uv_install(["huggingface-hub"])
from huggingface_hub import hf_hub_download

m = MODEL_CATALOG[MODEL_KEY]
print(f"다운로드: {MODEL_KEY} ({m['size']}) — {m['repo']}/{m['file']}")
print("최초 1회만 받고 이후 캐시됩니다. (대형 모델은 수 분 소요)")
gguf_path = hf_hub_download(repo_id=m["repo"], filename=m["file"])
os.environ["LLAMACPP_GGUF"] = gguf_path   # 기동 셀이 참조
print("GGUF 경로:", gguf_path)


[uv] 설치 완료: ['huggingface-hub']
다운로드: qwen2.5-0.5b (~0.4GB) — Qwen/Qwen2.5-0.5B-Instruct-GGUF/qwen2.5-0.5b-instruct-q4_k_m.gguf
최초 1회만 받고 이후 캐시됩니다. (대형 모델은 수 분 소요)


GGUF 경로: C:\Users\stshin\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct-GGUF\snapshots\9217f5db79a29953eb74d5343926648285ec7e67\qwen2.5-0.5b-instruct-q4_k_m.gguf


### 3-(3) llama.cpp 서버 기동 (주피터 백그라운드 · CPU)

아래 셀은 **현재 커널의 Python**(`sys.executable`)으로 `llama_cpp.server` 를
**백그라운드(`utils.run_cmd_bg`)** 로 띄웁니다. 백엔드가 `cpu` 이므로 `--n_gpu_layers 0` 으로 실행됩니다.
커널이 멈추지 않으며, 다른 셀 실행 중에도 `utils.tail_logs("llamacpp")` 로 로딩 로그를 확인할 수 있습니다.

In [9]:
# [기동] llama.cpp 서버를 백그라운드로 실행 (OpenAI 호환 /v1)
import os, sys
from urllib.parse import urlparse

port = urlparse(utils.LLAMACPP_BASE_URL).port or 8000
gguf = os.environ.get("LLAMACPP_GGUF", "")
# GPU 백엔드(cuXXX)면 전체 레이어를 GPU 로 오프로드(-1), CPU 면 0
n_gpu_layers = -1 if str(BACKEND).startswith("cu") else 0

if not gguf or not os.path.exists(gguf):
    print("[중단] GGUF 모델이 없습니다. 위 '모델 다운로드' 셀을 먼저 실행하세요.")
else:
    # 현재 커널의 venv python 으로 실행해야 llama_cpp 모듈이 정상 import 된다.
    py = sys.executable
    cmd = (
        f'"{py}" -m llama_cpp.server '
        f'--model "{gguf}" --model_alias {utils.LLAMACPP_MODEL} '
        f'--host 0.0.0.0 --port {port} --n_gpu_layers {n_gpu_layers} --n_ctx 4096'
    )
    print(f"백엔드={BACKEND} · n_gpu_layers={n_gpu_layers} · 모델={MODEL_KEY}")
    utils.run_cmd_bg(cmd, "llamacpp")
    print("\n서버 로딩까지 잠시 걸립니다(대형 모델일수록 오래). 다음 헬스체크 셀로 확인하세요.")
    print('중지하려면:  utils.stop_bg("llamacpp")')


백엔드=cpu · n_gpu_layers=0 · 모델=qwen2.5-0.5b
[백그라운드 시작] name='llamacpp'
$ "C:\Users\stshin\Documents\GitHub\Agentic AI Tutorial\.venv\Scripts\python.exe" -m llama_cpp.server --model "C:\Users\stshin\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct-GGUF\snapshots\9217f5db79a29953eb74d5343926648285ec7e67\qwen2.5-0.5b-instruct-q4_k_m.gguf" --model_alias llamacpp --host 0.0.0.0 --port 8000 --n_gpu_layers 0 --n_ctx 4096
  → 진행 로그는 다른 셀에서  utils.tail_logs('llamacpp')  로 확인하세요.

서버 로딩까지 잠시 걸립니다(대형 모델일수록 오래). 다음 헬스체크 셀로 확인하세요.
중지하려면:  utils.stop_bg("llamacpp")


In [10]:
# [경로 2] (1) llama.cpp 연결 테스트 — 서버 로그 확인 + OpenAI 호환 /v1/models
# (서버가 아직 로딩 중이면 잠시 후 이 셀을 다시 실행하세요.)
utils.tail_logs("llamacpp", 12)
print()
connection_test("llamacpp", utils.LLAMACPP_BASE_URL)

[llamacpp] 상태: 실행 중  ·  최근 12줄
------------------------------------------------------------

[llamacpp] 연결 테스트 → http://localhost:8000/v1/models


  ✅ 서버 응답 OK — 서빙 모델: ['llamacpp']


True

In [11]:
# [경로 2] (2) llama.cpp OpenAI 호환 프로토콜 응답 테스트 (CPU · 소형 GGUF)
llamacpp_llm = response_test("llamacpp")   # get_llm("llamacpp") → ChatOpenAI(:8000/v1)

[llamacpp] 응답 테스트 · LLM 타입=ChatOpenAI


  응답: 로컬 LLM을 OpenAI 호환 API로 쓰면, 사용자는 쉽게 사용할 수 있는 기능을 제공하고, 데이터베이스와 인공지능의 통신을 간결하게 처리할 수 있습니다.


---
## 4. 정리 — 두 경로 비교 (OpenAI 호환 프로토콜)

| 경로 | 서버 | 실행 | 모델 | 엔드포인트 |
|---|---|---|---|---|
| 경로 1 | **Ollama** | **GPU**(자동) | `qwen3:8b` | `:11434/v1` |
| 경로 2 | **llama.cpp** | **CPU**(고정) | 소형 GGUF | `:8000/v1` |

두 경로 모두 **같은 `ChatOpenAI` 코드**(`get_llm(provider).invoke(...)`)로 접속했고,
차이는 **백엔드(GPU/CPU)와 서버 종류**뿐입니다. 이것이 **OpenAI 호환 프로토콜**의 핵심 이점입니다 —
클라우드·로컬, GPU·CPU 를 **코드 변경 없이** 오갈 수 있습니다.

In [12]:
# [정리] 두 로컬 경로를 나란히 확인 — 연결 상태 + 각 서버가 서빙하는 모델
print("=== 두 로컬 LLM 경로 (OpenAI 호환 /v1) ===")
for prov, base, run in [("ollama",   utils.OLLAMA_BASE_URL,   "GPU"),
                        ("llamacpp", utils.LLAMACPP_BASE_URL, "CPU")]:
    ok, info = check_openai_server(base)
    status = f"✅ 서빙 {info}" if ok else f"❌ {info}"
    print(f"  경로: {prov:9s} [{run}]  {base:26s}  {status}")
print("\n두 경로 모두 동일한 ChatOpenAI 인터페이스로 접속 — 백엔드(GPU/CPU)만 다릅니다.")

=== 두 로컬 LLM 경로 (OpenAI 호환 /v1) ===


  경로: ollama    [GPU]  http://localhost:11434/v1   ✅ 서빙 ['qwen2.5:0.5b', 'nomic-embed-text:latest', 'qwen3:8b']


  경로: llamacpp  [CPU]  http://localhost:8000/v1    ✅ 서빙 ['llamacpp']

두 경로 모두 동일한 ChatOpenAI 인터페이스로 접속 — 백엔드(GPU/CPU)만 다릅니다.
